In [1]:
import cv2

In [1]:
import cv2
import numpy as np
import os


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    # Convert to grayscale if needed
    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Bilateral filtering
    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    # Canny edge detection
    edges = cv2.Canny(blur, 50, 120)

    # Morphological closing
    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# CONTOUR DRAWING
# =====================================================

def draw_large_contours(frame, edges, min_area=9000):

    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    output = frame.copy()

    kept_count = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

            kept_count += 1

    return output, kept_count


# =====================================================
# FRAME ENHANCEMENT PIPELINE
# =====================================================

def enhance_frame(frame):

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Denoising
    denoised = gaussian_denoise(gray)

    # Contrast stretching
    stretched = contrast_stretch(denoised)

    # Sharpening
    sharpened = sharpen_image(stretched)

    # Convert back to BGR
    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    # Open input video
    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    # Read video properties
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Video writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    if not writer.isOpened():
        print("Error creating output video.")
        return

    frames_processed = 0
    frames_with_segments = 0

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    # =================================================
    # FRAME PROCESSING LOOP
    # =================================================

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # Enhancement pipeline
        enhanced = enhance_frame(frame)

        # Edge detection pipeline
        edges = build_edges(enhanced)

        # Contour segmentation
        output_frame, kept_count = draw_large_contours(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        # Count segmented frames
        if kept_count > 0:
            frames_with_segments += 1

        # Save frame
        writer.write(output_frame)

        # Preview window
        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        frames_processed += 1

        # Press q to quit
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    # =================================================
    # RELEASE RESOURCES
    # =================================================

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("\nPipeline completed.")
    print("Frames processed:", frames_processed)
    print("Frames with segmentation:", frames_with_segments)
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation pipeline...
Press q to stop preview.

Pipeline completed.
Frames processed: 21
Frames with segmentation: 20
Output saved as: IMG_7555_segmented.mp4


In [4]:
import cv2
import numpy as np
import os


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    # Convert to grayscale if needed
    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Bilateral filtering
    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    # Canny edge detection
    edges = cv2.Canny(blur, 50, 120)

    # Morphological closing
    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# CONTOUR DRAWING
# =====================================================

def draw_large_contours(frame, edges, min_area=9000):

    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    output = frame.copy()

    kept_count = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

            kept_count += 1

    return output, kept_count


# =====================================================
# FRAME ENHANCEMENT PIPELINE
# =====================================================

def enhance_frame(frame):

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Denoising
    denoised = gaussian_denoise(gray)

    # Contrast stretching
    stretched = contrast_stretch(denoised)

    # Sharpening
    sharpened = sharpen_image(stretched)

    # Convert back to BGR
    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    # Open input video
    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    # Read video properties
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Video writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    if not writer.isOpened():
        print("Error creating output video.")
        return

    frames_processed = 0
    frames_with_segments = 0

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    # =================================================
    # FRAME PROCESSING LOOP
    # =================================================

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # Enhancement pipeline
        enhanced = enhance_frame(frame)

        # Edge detection pipeline
        edges = build_edges(enhanced)

        # Contour segmentation
        output_frame, kept_count = draw_large_contours(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        # =================================================
        # ADD ROI BOXES
        # =================================================

        # Get frame dimensions
        h, w = output_frame.shape[:2]

        # ROI dimensions
        roi_width = int(w * 0.22)
        roi_height = int(h * 0.18)

        # Bottom margin
        bottom_margin = 20

        # LEFT ROI (Light Green)
        left_x1 = int(w * 0.08)
        left_y1 = h - roi_height - bottom_margin
        left_x2 = left_x1 + roi_width
        left_y2 = h - bottom_margin

        cv2.rectangle(
            output_frame,
            (left_x1, left_y1),
            (left_x2, left_y2),
            (144, 238, 144),   # Light Green
            3
        )

        # CENTER ROI (Light Blue)
        center_x1 = (w // 2) - (roi_width // 2)
        center_y1 = h - roi_height - bottom_margin
        center_x2 = center_x1 + roi_width
        center_y2 = h - bottom_margin

        cv2.rectangle(
            output_frame,
            (center_x1, center_y1),
            (center_x2, center_y2),
            (255, 255, 0),     # Light Blue
            3
        )

        # RIGHT ROI (Light Green)
        right_x2 = int(w * 0.92)
        right_x1 = right_x2 - roi_width
        right_y1 = h - roi_height - bottom_margin
        right_y2 = h - bottom_margin

        cv2.rectangle(
            output_frame,
            (right_x1, right_y1),
            (right_x2, right_y2),
            (144, 238, 144),   # Light Green
            3
        )

        # Count segmented frames
        if kept_count > 0:
            frames_with_segments += 1

        # Save frame
        writer.write(output_frame)

        # Preview window
        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        frames_processed += 1

        # Press q to quit
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    # =================================================
    # RELEASE RESOURCES
    # =================================================

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("\nPipeline completed.")
    print("Frames processed:", frames_processed)
    print("Frames with segmentation:", frames_with_segments)
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation pipeline...
Press q to stop preview.

Pipeline completed.
Frames processed: 72
Frames with segmentation: 71
Output saved as: IMG_7555_segmented.mp4
